# Split Synthetic Images into Train and Test Sets

This notebook reads all CSV files in the dataset `Metadata` directory, uses `synthetic_split.csv` to determine each image's split, and copies the images into a new `Synthetic_split/train` and `Synthetic_split/test` directory. The original `Synthetic` directory is unchanged.

In [1]:
from pathlib import Path
import shutil
from IPython.display import display, Image
import json

import pandas as pd

# The notebook is expected to run from the repository root.
REPO_ROOT = Path.cwd()
DATASET_ROOT = (
    REPO_ROOT
    / "data"
    / "f5ddba02-4295-48d9-8ad5-ba8583871323"
    / "bc6ba36c-aa02-4630-a0ea-4150b9b69e36"
)
METADATA_DIR = DATASET_ROOT / "Metadata"
SYNTHETIC_DIR = DATASET_ROOT / "Synthetic"
OUTPUT_DIR = DATASET_ROOT / "Synthetic_split"

if not METADATA_DIR.is_dir():
    raise FileNotFoundError(f"Metadata directory not found: {METADATA_DIR}")
if not SYNTHETIC_DIR.is_dir():
    raise FileNotFoundError(f"Synthetic image directory not found: {SYNTHETIC_DIR}")

print(f"Dataset root: {DATASET_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Dataset root: /Users/jmaasch/Desktop/repos/object_detection/data/f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-aa02-4630-a0ea-4150b9b69e36
Output directory: /Users/jmaasch/Desktop/repos/object_detection/data/f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-aa02-4630-a0ea-4150b9b69e36/Synthetic_split


## Read Metadata CSV Files

All CSV files in `Metadata` are loaded into a dictionary keyed by filename stem. The split file is selected by its filename rather than relying on a particular CSV ordering.

In [2]:
metadata_files = sorted(METADATA_DIR.glob("*.csv"))
if not metadata_files:
    raise FileNotFoundError(f"No CSV files found in {METADATA_DIR}")

metadata_tables = {path.stem: pd.read_csv(path) for path in metadata_files}
print("Loaded metadata files:")
for name, table in metadata_tables.items():
    print(f"  {name}: {table.shape[0]:,} rows x {table.shape[1]} columns")

if "synthetic_split" not in metadata_tables:
    raise KeyError("Expected Metadata/synthetic_split.csv")

split_df = metadata_tables["synthetic_split"].copy()
required_columns = {"s3_path", "split"}
missing_columns = required_columns.difference(split_df.columns)
if missing_columns:
    raise KeyError(f"synthetic_split.csv is missing columns: {sorted(missing_columns)}")

split_df[["s3_path", "split"]].head()

Loaded metadata files:
  neighbors_metadata: 0 rows x 5 columns
  synthetic_metadata: 1,000 rows x 18 columns
  synthetic_split: 1,000 rows x 19 columns


,s3_path,split
0,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,train
1,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,train
2,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,train
3,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,train
4,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,train


In [3]:
split_df

,context_id,input_query,s3_path,environment_original,weather_condition_original,visibility_original,vehicle_type_original,vehicle_count_original,region_original,pavement_color_original,truck_count_original,truck_color_original,vehicle_color_original,image_angle_original,objects_of_interest_original,mask,s3_lora_uri,lora_trigger_word,split
0,3fa73564-7af2-424c-8355-04937d83a322,A foggy rural highway in the northeast USA wit...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,rural highways,foggy,hazy,SUVs,moderate number of vehicles,northeast USA,gray,one truck (Ford F150),red,varied colors,street-level,truck,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,train
1,e20a4238-de51-4369-830a-e994e1d574ce,An overhead view of a foggy urban highway in t...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,urban highways,foggy,hazy,SUVs,very few vehicles,southwest USA,gray,one truck (Ford F150),red,varied colors,overhead,car,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,train
2,7d448133-69ed-43a5-9deb-dd974814a651,A rural highway in the southwest USA with a hi...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,rural highways,sunny,hazy,trucks (Ford F150),high density of vehicles,southwest USA,gray,one truck (Ford F150),white,varied colors,street-level,motorcycle,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,train
3,d5f04948-4775-456e-86bf-169058df4c51,An urban highway in the southeast USA with gra...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,urban highways,rainy,hazy,motorcycles,very few vehicles,southeast USA,gray,one truck (Ford F150),blue,varied colors,street-level,car,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,train
4,c4b74337-92cd-431a-ad87-c699c6490fbf,An overhead view of a foggy urban highway in t...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,urban highways,foggy,clear,motorcycles,very few vehicles,southwest USA,red-brown,one truck (Ford F150),red,varied colors,overhead,truck,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,76e45796-4911-4bf5-a044-57ed38439378,An overhead view of foggy urban highways in th...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,urban highways,foggy,clear,trucks (Ford F150),moderate number of vehicles,northeast USA,gray,one truck (Ford F150),white,varied colors,overhead,motorcycle,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,test
996,f6f94313-0ec1-4b07-919f-cf1a0f04a1d6,An urban highway in the southeast USA with a s...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,urban highways,snowy,hazy,buses,moderate number of vehicles,southeast USA,red-brown,one truck (Ford F150),white,varied colors,street-level,bus,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,test
997,ac129be6-fb93-4bf1-bb5a-9f283405f23a,An urban highway in the southwest USA with fog...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,urban highways,foggy,clear,motorcycles,very few vehicles,southwest USA,gray,one truck (Ford F150),red,varied colors,street-level,bus,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,test
998,f949c8ef-fb4e-4097-8228-319055aedb0d,An overhead view of rural highways in the nort...,f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-...,rural highways,snowy,clear,buses,high density of vehicles,northeast USA,gray,one truck (Ford F150),red,varied colors,overhead,bus,NaN,s3://auto-lora/f5ddba02-4295-48d9-8ad5-ba85838...,$txc,test


In [4]:
# Normalize split labels and use only the image basename from each metadata path.
split_df["split"] = split_df["split"].astype(str).str.strip().str.lower()
split_df["image_name"] = split_df["s3_path"].map(lambda value: Path(str(value)).name)

allowed_splits = {"train", "test"}
unexpected_splits = sorted(set(split_df["split"]) - allowed_splits)
if unexpected_splits:
    raise ValueError(f"Unexpected split labels: {unexpected_splits}. Expected train/test.")

duplicate_names = split_df["image_name"][split_df["image_name"].duplicated()].unique().tolist()
if duplicate_names:
    raise ValueError(f"Images assigned more than once in synthetic_split.csv: {duplicate_names[:5]}")

missing_images = [
    image_name
    for image_name in split_df["image_name"]
    if not (SYNTHETIC_DIR / image_name).is_file()
]
if missing_images:
    raise FileNotFoundError(
        f"{len(missing_images)} images from synthetic_split.csv were not found in Synthetic."
        f" Examples: {missing_images[:5]}"
    )

print(split_df["split"].value_counts().sort_index())
print(f"Validated {len(split_df):,} split assignments and confirmed all source images exist.")

split
test     200
train    800
Name: count, dtype: int64
Validated 1,000 split assignments and confirmed all source images exist.


## Copy Images into Train and Test Directories

The output directory is created separately from the source directory. Existing files with the same names are overwritten so the notebook can be rerun safely.

In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

copied_counts = {split_name: 0 for split_name in sorted(allowed_splits)}
for row in split_df.itertuples(index=False):
    source_path = SYNTHETIC_DIR / row.image_name
    destination_dir = OUTPUT_DIR / row.split
    destination_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, destination_dir / row.image_name)
    copied_counts[row.split] += 1

print("Copied images:")
for split_name, count in copied_counts.items():
    print(f"  {split_name}: {count:,}")

Copied images:
  test: 200
  train: 800


In [6]:
# Verify the output counts and report the resulting directories.
output_counts = {
    split_name: len(list((OUTPUT_DIR / split_name).glob("*")))
    for split_name in sorted(allowed_splits)
}

if output_counts != copied_counts:
    raise RuntimeError(f"Output verification failed: expected {copied_counts}, found {output_counts}")

print("Split complete and verified.")
print(f"Train images: {OUTPUT_DIR / 'train'}")
print(f"Test images:  {OUTPUT_DIR / 'test'}")

Split complete and verified.
Train images: /Users/jmaasch/Desktop/repos/object_detection/data/f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-aa02-4630-a0ea-4150b9b69e36/Synthetic_split/train
Test images:  /Users/jmaasch/Desktop/repos/object_detection/data/f5ddba02-4295-48d9-8ad5-ba8583871323/bc6ba36c-aa02-4630-a0ea-4150b9b69e36/Synthetic_split/test


# End of document